# CS 5326: Advanced Generative AI and Agents — Programming Assignment 3
## Search over Reasoning and Step-Level Process Reward Models

This notebook is **100% self-contained** and runs top-to-bottom on **Kaggle GPU T4x2** (or locally):
1. Sets up the repository structure and upgrades `bitsandbytes` on Kaggle.
2. Writes the complete implementation modules (`src/*.py`, `run_all.py`, `analyze_outputs.py`).
3. Executes the offline public smoke tests and token-counting tests.
4. Loads the asymmetric policy-verifier pair (`Qwen2.5-1.5B-Instruct` in `float16` on `cuda:0` and `Qwen2.5-7B-Instruct` in 4-bit `NF4` on `cuda:1`).
5. Runs the 50-problem GSM8K benchmark with incremental per-problem checkpointing (`outputs/per_problem.jsonl`).
6. Serializes all required evaluation artifacts (`outputs/results_summary.json`, `outputs/aggregation_ablation.json`, `outputs/sc_scaling_curve.json`, `outputs/per_problem.jsonl`), generates plots, and bundles `pa3_kaggle_outputs.zip` for easy download.

### Step 1: Kaggle Environment & Starter Repository Setup

In [ ]:
import os
import subprocess
import sys

# If running on Kaggle without the dataset present, clone repository and upgrade bitsandbytes
if not os.path.exists("data/gsm8k_test_50.jsonl"):
    if not os.path.exists("cs5326-pa3"):
        res = subprocess.run(["git", "clone", "https://github.com/Ammar946/cs5326-pa3.git"], check=False)
        if res.returncode == 0:
            os.chdir("cs5326-pa3")
        else:
            subprocess.run(["git", "clone", "https://github.com/ahmad-sarfraz345/cs5326-pa3-starter.git"], check=True)
            os.chdir("cs5326-pa3-starter")
    else:
        os.chdir("cs5326-pa3")
    print("Working directory set to:", os.getcwd())

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or not os.environ.get("SKIP_BNB_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=False)


### Step 2: Write Implementation Modules (`src/*.py`, `run_all.py`, `analyze_outputs.py`)

In [ ]:
import os
os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("figures", exist_ok=True)

FILES_TO_WRITE = {
    "src/extraction.py": '"""\nsrc/extraction.py - Step and answer extraction utilities.\n"""\nimport re\n\n\ndef parse_steps(completion_text: str) -> list[str]:\n    """\n    Parses completion text and returns a list of individual step strings.\n    Must tolerate markdown decoration around labels (e.g. \'**Step 1:**\', \'### Step 2:\')\n    and strip leftover \'*\' or \'#\' from captured text.\n    """\n    if not completion_text or not isinstance(completion_text, str):\n        return []\n\n    # Pattern matching Step markers with optional markdown decoration (#, *, `, :, .)\n    pattern = (\n        r"(?:^|\\n)\\s*(?:[#*`\\s]*Step\\s*\\d+\\s*[:.]?[#*`\\s]*)(.*?)"\n        r"(?=(?:\\n\\s*[#*`\\s]*Step\\s*\\d+\\s*[:.]?[#*`\\s]*)|(?:\\n\\s*[#*`\\s]*Final\\s+Answer)|$)"\n    )\n    matches = re.findall(pattern, completion_text, flags=re.DOTALL | re.IGNORECASE)\n    cleaned = []\n    for m in matches:\n        text = m.strip().strip("*#`").strip()\n        if text:\n            cleaned.append(text)\n    return cleaned\n\n\ndef _parse_int_from_tail(tail: str) -> int | None:\n    """\n    Parses an integer value from the text immediately following a \'Final Answer\' delimiter.\n    Returns None if the value is non-integer (e.g. 18.5) or malformed.\n    """\n    if not tail or not isinstance(tail, str):\n        return None\n\n    cleaned_tail = tail.strip()\n    if not cleaned_tail:\n        return None\n\n    # Allow optional leading markdown (*, `), LaTeX $/ \\boxed{, sign, currency ($), commas, and decimal part\n    m = re.match(\n        r"^[\\s*`\\\\$]*(?:boxed\\{\\s*)?[\\s*`\\\\]*([+-])?\\s*\\$?\\s*([+-])?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(\\.\\d+)?",\n        cleaned_tail,\n    )\n    if not m:\n        return None\n\n    sign1, sign2, int_part, dec_part = m.group(1), m.group(2), m.group(3), m.group(4)\n    # Disallow conflicting double signs like \'+-5\' or \'--5\'\n    if sign1 and sign2:\n        return None\n\n    # Ensure the matched number is not immediately followed by another decimal point or fraction slash + digit\n    after_pos = m.end()\n    rest = cleaned_tail[after_pos:]\n    if re.match(r"^\\s*[/\\.]\\s*\\d", rest):\n        return None\n\n    # If a decimal part is present, it must be strictly zero (e.g. \'.00\' is valid, \'.5\' is None)\n    if dec_part is not None:\n        frac_digits = dec_part[1:]\n        if not frac_digits or any(ch != "0" for ch in frac_digits):\n            return None\n\n    sign = -1 if (sign1 == "-" or sign2 == "-") else 1\n    try:\n        value = int(int_part.replace(",", ""))\n        return sign * value\n    except (ValueError, TypeError):\n        return None\n\n\ndef extract_answer(completion_text: str) -> int | None:\n    """\n    Extracts the final numeric integer from the completion.\n\n    Required Behaviors:\n      - \'Final Answer: 18\' -> 18\n      - \'Final Answer: $1,234\' -> 1234\n      - \'Final Answer: 18.00\' -> 18\n      - \'Final Answer: 18.5\' -> None (non-integer)\n      - \'Final Answer: -5\' -> -5\n      - \'Final Answer: 18 dollars.\' -> 18\n      - \'**Final Answer:** 18\' -> 18\n      - \'final answer 18\' (case-insensitive, colon optional) -> 18\n      - Multiple \'Final Answer\' lines -> use the LAST one\n      - No \'Final Answer\' present -> None\n      - Empty string or None-like garbage -> None, and never raise\n    """\n    try:\n        if not completion_text or not isinstance(completion_text, str):\n            return None\n\n        # Primary pattern: \'Final Answer\' at start of string or start of line (colon optional)\n        line_pattern = r"(?:^|\\n)\\s*[#*`\\s]*final\\s+answer\\b[#*`\\s]*[:=]?\\s*([^\\n\\r]*)"\n        matches = re.findall(line_pattern, completion_text, flags=re.IGNORECASE)\n\n        if not matches:\n            # Secondary pattern: inline \'Final Answer:\' with explicit colon/equals\n            inline_pattern = r"[#*`\\s]*final\\s+answer[#*`\\s]*[:=]\\s*([^\\n\\r]*)"\n            matches = re.findall(inline_pattern, completion_text, flags=re.IGNORECASE)\n\n        if not matches:\n            return None\n\n        # Always inspect the LAST \'Final Answer\' occurrence\n        last_tail = matches[-1]\n        return _parse_int_from_tail(last_tail)\n    except Exception:\n        return None\n',
    "src/generation.py": '"""\nsrc/generation.py - Prompt formatting and baseline generation routines.\n"""\nimport time\nfrom typing import Any, Callable\n\ntry:\n    from .extraction import extract_answer, parse_steps\n    from .utils import GREEDY_GEN_KWARGS, SAMPLING_GEN_KWARGS, make_hf_generate_fn\nexcept ImportError:\n    from src.extraction import extract_answer, parse_steps\n    from src.utils import GREEDY_GEN_KWARGS, SAMPLING_GEN_KWARGS, make_hf_generate_fn\n\nGenerateFn = Callable[[str, int, dict], list[dict]]\n# generate_fn(raw_prompt, num_samples, gen_kwargs) -> list of {"text": str, "num_tokens": int}\n# gen_kwargs carries do_sample/temperature/top_p/top_k/repetition_penalty/max_new_tokens\n\n\ndef format_step_prompt(question: str) -> str:\n    """\n    Formats the raw problem question into a prompt that instructs the \n    model to emit numbered steps (\'Step 1: ...\') and \'Final Answer: <int>\'.\n    \n    Returns a raw instruction string (no chat template applied).\n    """\n    q_text = question.strip() if isinstance(question, str) else str(question)\n    return (\n        "Solve the following math word problem step by step.\\n"\n        "You must strictly follow this exact output format:\\n"\n        "1. Write each deduction or calculation as a separate numbered step starting with "\n        "\'Step 1:\', \'Step 2:\', \'Step 3:\', etc. on its own line.\\n"\n        "2. Do NOT use markdown headings (###), bullet points, LaTeX formatting, or \\\\boxed{}.\\n"\n        "3. Conclude your solution on a new final line in the exact format:\\n"\n        "Final Answer: <integer>\\n"\n        "Write only the plain integer after \'Final Answer:\' (no units, commas, or currency symbols).\\n\\n"\n        "Example format:\\n"\n        "Step 1: Compute the initial amount: 3 + 4 = 7.\\n"\n        "Step 2: Multiply by 2 to find the total: 7 * 2 = 14.\\n"\n        "Final Answer: 14\\n\\n"\n        f"Question: {q_text}\\n"\n    )\n\n\ndef compute_majority_vote(answers: list[int | None]) -> int | None:\n    """\n    Given a list of extracted answers, returns the majority answer.\n    Ignores None values. Breaks ties deterministically in favor of the answer\n    that appeared earliest in the list.\n    """\n    if not answers:\n        return None\n\n    counts: dict[int, int] = {}\n    first_index: dict[int, int] = {}\n\n    for idx, ans in enumerate(answers):\n        if ans is None:\n            continue\n        if ans not in counts:\n            counts[ans] = 0\n            first_index[ans] = idx\n        counts[ans] += 1\n\n    if not counts:\n        return None\n\n    # Maximize (frequency, -first_occurrence_index)\n    best_ans = max(counts.keys(), key=lambda a: (counts[a], -first_index[a]))\n    return best_ans\n\n\ndef run_greedy(\n    model: Any = None, \n    tokenizer: Any = None, \n    question: str = "", \n    generate_fn: GenerateFn | None = None\n) -> dict[str, Any]:\n    """\n    Executes greedy decoding for a single question.\n    Calls generate_fn with num_samples=1 and gen_kwargs containing do_sample=False,\n    temperature=None, top_p=None, top_k=None.\n    \n    Returns:\n      prompt, completion, steps, answer, generation_calls, generated_tokens, time_seconds\n    """\n    t0 = time.perf_counter()\n    prompt = format_step_prompt(question)\n\n    if generate_fn is None:\n        if model is None or tokenizer is None:\n            raise ValueError("Either generate_fn or (model, tokenizer) must be provided to run_greedy.")\n        generate_fn = make_hf_generate_fn(model, tokenizer)\n\n    gen_kwargs = dict(GREEDY_GEN_KWARGS)\n    outputs = generate_fn(prompt, 1, gen_kwargs)\n\n    if outputs:\n        completion = outputs[0].get("text", "")\n        generated_tokens = int(outputs[0].get("num_tokens", 0))\n    else:\n        completion = ""\n        generated_tokens = 0\n\n    steps = parse_steps(completion)\n    answer = extract_answer(completion)\n    elapsed = time.perf_counter() - t0\n\n    return {\n        "prompt": prompt,\n        "completion": completion,\n        "steps": steps,\n        "answer": answer,\n        "generation_calls": 1,\n        "generated_tokens": generated_tokens,\n        "time_seconds": float(elapsed),\n    }\n\n\ndef run_self_consistency(\n    model: Any = None, \n    tokenizer: Any = None, \n    question: str = "", \n    n: int = 5, \n    temperature: float = 0.7, \n    generate_fn: GenerateFn | None = None\n) -> dict[str, Any]:\n    """\n    Samples n diverse reasoning trajectories and aggregates candidate answers via majority vote.\n    \n    Returns:\n      candidates [each: completion, steps, answer, num_tokens], answer, \n      generation_calls, generated_tokens, time_seconds\n    """\n    t0 = time.perf_counter()\n    if n <= 0:\n        return {\n            "candidates": [],\n            "answer": None,\n            "generation_calls": 0,\n            "generated_tokens": 0,\n            "time_seconds": float(time.perf_counter() - t0),\n        }\n\n    prompt = format_step_prompt(question)\n\n    if generate_fn is None:\n        if model is None or tokenizer is None:\n            raise ValueError("Either generate_fn or (model, tokenizer) must be provided to run_self_consistency.")\n        generate_fn = make_hf_generate_fn(model, tokenizer)\n\n    gen_kwargs = dict(SAMPLING_GEN_KWARGS)\n    gen_kwargs["temperature"] = temperature\n\n    outputs = generate_fn(prompt, n, gen_kwargs)\n\n    candidates: list[dict[str, Any]] = []\n    for item in outputs:\n        comp_text = item.get("text", "")\n        num_toks = int(item.get("num_tokens", 0))\n        steps = parse_steps(comp_text)\n        ans = extract_answer(comp_text)\n        candidates.append({\n            "completion": comp_text,\n            "steps": steps,\n            "answer": ans,\n            "num_tokens": num_toks,\n        })\n\n    voted_answer = compute_majority_vote([c["answer"] for c in candidates])\n    total_tokens = sum(c["num_tokens"] for c in candidates)\n    elapsed = time.perf_counter() - t0\n\n    return {\n        "candidates": candidates,\n        "answer": voted_answer,\n        "generation_calls": len(candidates),\n        "generated_tokens": total_tokens,\n        "time_seconds": float(elapsed),\n    }\n',
    "src/prm_judge.py": '"""\nsrc/prm_judge.py - PRM judge prompt construction, parsing, and step scoring.\n"""\nimport math\nimport re\nfrom typing import Any, Callable\n\ntry:\n    from .utils import make_hf_judge_fn\nexcept ImportError:\n    from src.utils import make_hf_judge_fn\n\nJudgeFn = Callable[[str], dict]\n# judge_fn(judge_prompt) -> {"text": str, "num_tokens": int}\n\n\ndef build_prm_judge_prompt(question: str, preceding_steps: list[str], current_step: str) -> str:\n    """\n    Builds the verification prompt for the PRM judge to classify current_step\n    as VALID or INVALID given the problem question and preceding_steps.\n    \n    Judge Semantics (Local Validity):\n    The judge evaluates current_step ASSUMING ALL PRECEDING STEPS ARE CORRECT.\n    A step is VALID if its arithmetic is sound and it follows logically from the\n    problem statement plus preceding steps as written. An error in an earlier step\n    does NOT make subsequent steps invalid by itself.\n    \n    Autograder Literal Checks:\n    - Preceding steps are rendered as \'Step 1: <text>\', \'Step 2: <text>\', ... in order.\n    - The current step is rendered with the label \'Current Step:\'.\n    - The prompt contains the phrase \'assume all preceding steps are correct\' (case-insensitive).\n    - The prompt contains the literal strings \'Verdict: VALID\', \'Verdict: INVALID\', and \'Rationale:\'.\n    - With an empty preceding_steps list, the prompt states \'(no preceding steps)\'.\n    """\n    q_text = question.strip() if isinstance(question, str) else str(question)\n    curr_text = current_step.strip() if isinstance(current_step, str) else str(current_step)\n\n    if preceding_steps:\n        rendered_steps = []\n        for idx, step in enumerate(preceding_steps, start=1):\n            step_str = str(step).strip()\n            # Avoid duplicating \'Step K:\' if caller passed already-labeled step text\n            step_clean = re.sub(r"^\\s*Step\\s*\\d+\\s*[:.]?\\s*", "", step_str, flags=re.IGNORECASE)\n            rendered_steps.append(f"Step {idx}: {step_clean}")\n        preceding_block = "\\n".join(rendered_steps)\n    else:\n        preceding_block = "(no preceding steps)"\n\n    return (\n        "You are an expert mathematical reasoning verifier.\\n"\n        "Your task is to evaluate ONLY the Current Step for logical and arithmetic soundness.\\n"\n        "Local Validity Rule: You must assume all preceding steps are correct as written, "\n        "even if an earlier step contained a mistake. Judge whether the Current Step "\n        "follows logically from the Problem and Preceding Steps and whether its own "\n        "arithmetic calculation is accurate.\\n\\n"\n        f"Problem:\\n{q_text}\\n\\n"\n        f"Preceding Steps:\\n{preceding_block}\\n\\n"\n        f"Current Step:\\n{curr_text}\\n\\n"\n        "Verification Instructions:\\n"\n        "- If the Current Step is mathematically and logically sound (when you assume all preceding steps are correct), output \'Verdict: VALID\'.\\n"\n        "- If the Current Step contains an arithmetic calculation error or logical fallacy, output \'Verdict: INVALID\'.\\n"\n        "- Provide a single concise sentence after \'Rationale:\' explaining your decision.\\n\\n"\n        "Respond in the exact two-line format below:\\n"\n        "Verdict: <VALID or INVALID>\\n"\n        "Rationale: <one concise sentence explaining the verification decision>"\n    )\n\n\ndef parse_judge_decision(judge_output: str) -> tuple[float, str]:\n    r"""\n    Parses the judge model\'s raw text completion into (score, rationale).\n    \n    Rules:\n      1. If a line matches /verdict\\s*:\\s*\\**\\s*(VALID|INVALID)\\b/i, use it.\n         If several exist, use the first.\n      2. Otherwise, if standalone word INVALID or phrase \'NOT VALID\' appears\n         (case-insensitive, word boundaries) -> -1.0.\n      3. Otherwise, if standalone word VALID appears -> +1.0.\n      4. Otherwise -> -1.0 (unparseable defaults to INVALID).\n      \n    Warning: A naive \'"VALID" in text\' check is wrong because "INVALID" contains "VALID".\n    \n    Rationale: Text after \'Rationale:\' up to the first sentence end (".", "!", or "?"\n    followed by whitespace or end of string, so decimals like "2.5" are not split),\n    or a newline, whichever comes first, stripped. If absent, returns empty string "".\n    """\n    if not judge_output or not isinstance(judge_output, str):\n        return -1.0, ""\n\n    # 1. Verdict extraction\n    verdict_match = re.search(\n        r"verdict\\s*:\\s*\\**\\s*(VALID|INVALID)\\b",\n        judge_output,\n        flags=re.IGNORECASE,\n    )\n    if verdict_match:\n        token = verdict_match.group(1).upper()\n        score = 1.0 if token == "VALID" else -1.0\n    elif re.search(r"\\b(?:INVALID|NOT\\s+VALID)\\b", judge_output, flags=re.IGNORECASE):\n        score = -1.0\n    elif re.search(r"\\bVALID\\b", judge_output, flags=re.IGNORECASE):\n        score = 1.0\n    else:\n        score = -1.0\n\n    # 2. Rationale extraction\n    rationale = ""\n    rat_match = re.search(r"rationale\\s*:\\s*\\**\\s*([^\\n\\r]*)", judge_output, flags=re.IGNORECASE)\n    if rat_match:\n        line_tail = rat_match.group(1).strip()\n        if line_tail:\n            # Find first sentence end (\'.\', \'!\', or \'?\' followed by whitespace or end of string)\n            sent_end = re.search(r"[.!?](?=\\s|$)", line_tail)\n            if sent_end:\n                rationale = line_tail[: sent_end.end()].strip()\n            else:\n                rationale = line_tail.strip()\n            rationale = rationale.strip("*").strip()\n\n    return float(score), rationale\n\n\ndef score_path(\n    judge_model: Any = None, \n    judge_tokenizer: Any = None, \n    question: str = "", \n    candidate: dict[str, Any] | None = None, \n    judge_fn: JudgeFn | None = None\n) -> dict[str, Any]:\n    """\n    Sequentially scores each step in candidate["steps"] using judge_fn.\n    Step t is judged with preceding_steps = steps[:t].\n    \n    The student\'s code MUST call build_prm_judge_prompt to build the string passed to judge_fn,\n    and MUST call parse_judge_decision on the returned text.\n    \n    Returns:\n      step_scores, rationales, judge_calls, judge_tokens\n      (plus \'step_probs\' if judge_fn provides continuous logprob \'valid_prob\')\n    """\n    if candidate is None:\n        steps: list[str] = []\n    else:\n        steps = list(candidate.get("steps") or [])\n\n    if not steps:\n        return {\n            "step_scores": [],\n            "rationales": [],\n            "judge_calls": 0,\n            "judge_tokens": 0,\n        }\n\n    if judge_fn is None:\n        if judge_model is None or judge_tokenizer is None:\n            raise ValueError("Either judge_fn or (judge_model, judge_tokenizer) must be provided to score_path.")\n        judge_fn = make_hf_judge_fn(judge_model, judge_tokenizer)\n\n    step_scores: list[float] = []\n    rationales: list[str] = []\n    step_probs: list[float] = []\n    has_probs = False\n    judge_calls = 0\n    judge_tokens = 0\n\n    for t, current_step in enumerate(steps):\n        preceding_steps = steps[:t]\n        prompt = build_prm_judge_prompt(question, preceding_steps, current_step)\n        res = judge_fn(prompt)\n\n        raw_text = res.get("text", "")\n        num_toks = int(res.get("num_tokens", 0))\n        score, rationale = parse_judge_decision(raw_text)\n\n        step_scores.append(float(score))\n        rationales.append(rationale)\n        judge_calls += 1\n        judge_tokens += num_toks\n\n        if "valid_prob" in res and res["valid_prob"] is not None:\n            has_probs = True\n            step_probs.append(float(res["valid_prob"]))\n        else:\n            # Fallback clamped probability from discrete score\n            step_probs.append(0.99 if score > 0 else 0.01)\n\n    result: dict[str, Any] = {\n        "step_scores": step_scores,\n        "rationales": rationales,\n        "judge_calls": judge_calls,\n        "judge_tokens": judge_tokens,\n    }\n    if has_probs:\n        result["step_probs"] = step_probs\n    return result\n',
    "src/search.py": '"""\nsrc/search.py - Path aggregation, Best-of-N search, and compute-matched search.\n"""\nimport copy\nimport time\nfrom typing import Any, Callable\n\ntry:\n    from .generation import compute_majority_vote, run_self_consistency\n    from .prm_judge import score_path\nexcept ImportError:\n    from src.generation import compute_majority_vote, run_self_consistency\n    from src.prm_judge import score_path\n\nGenerateFn = Callable[[str, int, dict], list[dict]]\nJudgeFn = Callable[[str], dict]\n\n\ndef aggregate_path_score(\n    step_scores: list[float], \n    strategy: str = "min", \n    penalty_lambda: float = 2.0\n) -> float:\n    """\n    Aggregates a list of step scores (+1.0 or -1.0, or continuous probabilities for \'product_logprob\')\n    into a scalar trajectory score.\n    \n    Rules:\n      - An empty list returns -1.0 for every strategy.\n      - \'min\': min(step_scores).\n      - \'product\': p_t = clamp((r_t + 1)/2, 0.01, 0.99); returns the raw product.\n      - \'step_penalty\': mean(step_scores) - penalty_lambda * count(r_t == -1.0).\n      - \'product_logprob\' (Optional Bonus): product of continuous step validity probabilities p_t in (0, 1).\n      - An unknown strategy raises ValueError.\n    """\n    valid_strategies = {"min", "product", "step_penalty", "product_logprob"}\n    if strategy not in valid_strategies:\n        raise ValueError(f"Unknown aggregation strategy: \'{strategy}\'. Must be one of {sorted(valid_strategies)}.")\n\n    if not step_scores:\n        return -1.0\n\n    if strategy == "min":\n        return float(min(step_scores))\n\n    if strategy == "product":\n        prod = 1.0\n        for r in step_scores:\n            p_t = min(0.99, max(0.01, (float(r) + 1.0) / 2.0))\n            prod *= p_t\n        return float(prod)\n\n    if strategy == "step_penalty":\n        n_steps = len(step_scores)\n        mean_score = sum(float(r) for r in step_scores) / n_steps\n        num_invalid = sum(1 for r in step_scores if float(r) == -1.0)\n        return float(mean_score - float(penalty_lambda) * num_invalid)\n\n    if strategy == "product_logprob":\n        prod = 1.0\n        for s in step_scores:\n            val = float(s)\n            # If caller passed discrete {-1.0, +1.0} instead of continuous probabilities in [0, 1]\n            if val < 0.0:\n                p_t = min(0.9999, max(1e-4, (val + 1.0) / 2.0))\n            else:\n                p_t = min(0.9999, max(1e-4, val))\n            prod *= p_t\n        return float(prod)\n\n    raise ValueError(f"Unknown aggregation strategy: \'{strategy}\'")\n\n\ndef select_best_path(\n    candidate_paths: list[dict[str, Any]], \n    aggregation_strategy: str = "min", \n    penalty_lambda: float = 2.0\n) -> dict[str, Any]:\n    """\n    Selects the winning candidate path from candidate_paths.\n    \n    Input Dictionary Contract:\n      Each candidate dictionary in candidate_paths must contain \'step_scores\' (list[float])\n      and \'answer\' (int | None). Any other keys present in the candidate dictionary\n      are passed through unchanged in the returned copy.\n    \n    Selection & Tie-Breaking Rules:\n      1. Compute path_score for each candidate with aggregate_path_score.\n         A candidate with an empty step_scores list gets -1.0.\n      2. Eligibility: if at least one candidate has a non-None answer, candidates\n         with answer=None are excluded. If all answers are None, all remain eligible.\n      3. Find maximum path_score among eligible candidates (tolerance 1e-9).\n         The tied set is all eligible candidates within tolerance of the maximum.\n      4. If the tied set has more than one candidate, compute compute_majority_vote\n         over their answers. Return the lowest-index candidate in the tied set whose\n         answer equals the vote. If the vote is None, return the lowest-index tied candidate.\n      5. Returns a shallow COPY of the chosen dict with two added keys:\n         \'path_score\' (float) and \'num_tied\' (int, size of the tied set).\n         Does not mutate the inputs.\n    """\n    if not candidate_paths:\n        raise ValueError("candidate_paths must not be empty.")\n\n    # 1. Compute path_score for each candidate\n    path_scores: list[float] = []\n    for cand in candidate_paths:\n        steps = cand.get("steps")\n        if steps is not None and len(steps) == 0:\n            path_scores.append(-1.0)\n            continue\n\n        if aggregation_strategy == "product_logprob" and cand.get("step_probs"):\n            raw_scores = list(cand.get("step_probs") or [])\n        else:\n            raw_scores = list(cand.get("step_scores") or [])\n\n        if not raw_scores:\n            path_scores.append(-1.0)\n        else:\n            score = aggregate_path_score(\n                raw_scores,\n                strategy=aggregation_strategy,\n                penalty_lambda=penalty_lambda,\n            )\n            path_scores.append(float(score))\n\n    # 2. Eligibility: exclude answer=None if at least one candidate has non-None answer\n    non_none_indices = [i for i, c in enumerate(candidate_paths) if c.get("answer") is not None]\n    if non_none_indices:\n        eligible_indices = non_none_indices\n    else:\n        eligible_indices = list(range(len(candidate_paths)))\n\n    # 3. Maximum path_score among eligible candidates (tolerance 1e-9)\n    max_score = max(path_scores[i] for i in eligible_indices)\n    tied_indices = [i for i in eligible_indices if abs(path_scores[i] - max_score) <= 1e-9]\n    num_tied = len(tied_indices)\n\n    # 4. Tie-breaking via majority vote over tied candidates\' answers\n    if num_tied == 1:\n        chosen_idx = tied_indices[0]\n    else:\n        tied_answers = [candidate_paths[i].get("answer") for i in tied_indices]\n        voted_ans = compute_majority_vote(tied_answers)\n        if voted_ans is None:\n            chosen_idx = tied_indices[0]\n        else:\n            chosen_idx = next(\n                (i for i in tied_indices if candidate_paths[i].get("answer") == voted_ans),\n                tied_indices[0],\n            )\n\n    # 5. Return a shallow copy of the chosen candidate dict with \'path_score\' and \'num_tied\'\n    selected_copy = copy.copy(candidate_paths[chosen_idx])\n    selected_copy["path_score"] = float(path_scores[chosen_idx])\n    selected_copy["num_tied"] = int(num_tied)\n    return selected_copy\n\n\ndef run_prm_best_of_n(\n    model: Any = None, \n    judge_model: Any = None, \n    gen_tokenizer: Any = None, \n    judge_tokenizer: Any = None, \n    question: str = "", \n    n: int = 5, \n    aggregation_strategy: str = "min", \n    penalty_lambda: float = 2.0, \n    candidates: list[dict[str, Any]] | None = None, \n    generate_fn: GenerateFn | None = None, \n    judge_fn: JudgeFn | None = None\n) -> dict[str, Any]:\n    """\n    Executes PRM-guided Best-of-N search over reasoning trajectories.\n    \n    If candidates is provided (shared candidate pool), does NOT sample new candidates\n    (generation_calls = 0, generated_tokens = 0). Provided candidates use the\n    run_self_consistency candidate format (at minimum \'steps\' and \'answer\').\n    Otherwise, samples n candidates using run_self_consistency.\n    \n    Scores each candidate\'s steps using score_path.\n    Selects the winning candidate via select_best_path.\n    \n    Returns:\n      selected, answer, candidates [each augmented with step_scores, rationales],\n      generation_calls, judge_calls, generated_tokens, judge_tokens, time_seconds\n    """\n    t0 = time.perf_counter()\n\n    if candidates is None:\n        sc_res = run_self_consistency(\n            model=model,\n            tokenizer=gen_tokenizer,\n            question=question,\n            n=n,\n            temperature=0.7,\n            generate_fn=generate_fn,\n        )\n        cand_list = [copy.copy(c) for c in sc_res["candidates"]]\n        generation_calls = int(sc_res["generation_calls"])\n        generated_tokens = int(sc_res["generated_tokens"])\n    else:\n        cand_list = [copy.copy(c) for c in candidates]\n        generation_calls = 0\n        generated_tokens = 0\n\n    total_judge_calls = 0\n    total_judge_tokens = 0\n    augmented_candidates: list[dict[str, Any]] = []\n\n    for cand in cand_list:\n        cand_copy = copy.copy(cand)\n        scored = score_path(\n            judge_model=judge_model,\n            judge_tokenizer=judge_tokenizer,\n            question=question,\n            candidate=cand_copy,\n            judge_fn=judge_fn,\n        )\n        cand_copy["step_scores"] = scored["step_scores"]\n        cand_copy["rationales"] = scored["rationales"]\n        if "step_probs" in scored:\n            cand_copy["step_probs"] = scored["step_probs"]\n\n        total_judge_calls += int(scored["judge_calls"])\n        total_judge_tokens += int(scored["judge_tokens"])\n        augmented_candidates.append(cand_copy)\n\n    selected = select_best_path(\n        augmented_candidates,\n        aggregation_strategy=aggregation_strategy,\n        penalty_lambda=penalty_lambda,\n    )\n    elapsed = time.perf_counter() - t0\n\n    return {\n        "selected": selected,\n        "answer": selected.get("answer"),\n        "candidates": augmented_candidates,\n        "generation_calls": generation_calls,\n        "judge_calls": total_judge_calls,\n        "generated_tokens": generated_tokens,\n        "judge_tokens": total_judge_tokens,\n        "time_seconds": float(elapsed),\n    }\n\n\ndef run_compute_matched_self_consistency(\n    model: Any = None, \n    tokenizer: Any = None, \n    question: str = "", \n    target_calls: int = 15, \n    temperature: float = 0.7, \n    existing_candidates: list[dict[str, Any]] | None = None, \n    generate_fn: GenerateFn | None = None\n) -> dict[str, Any]:\n    """\n    Executes compute-matched Self-Consistency aggregating target_calls paths.\n    If existing_candidates are supplied, samples only the deficit:\n    target_calls - len(existing_candidates). Returned candidates = existing_candidates followed by new samples, in sampling order.\n    \n    Returns:\n      candidates, answer, generation_calls (new samples only), total_samples,\n      generated_tokens (new samples only), time_seconds\n    """\n    t0 = time.perf_counter()\n\n    existing = [copy.copy(c) for c in existing_candidates] if existing_candidates else []\n    deficit = max(0, target_calls - len(existing))\n\n    if deficit > 0:\n        new_sc = run_self_consistency(\n            model=model,\n            tokenizer=tokenizer,\n            question=question,\n            n=deficit,\n            temperature=temperature,\n            generate_fn=generate_fn,\n        )\n        new_cands = new_sc["candidates"]\n        generation_calls = int(new_sc["generation_calls"])\n        generated_tokens = int(new_sc["generated_tokens"])\n    else:\n        new_cands = []\n        generation_calls = 0\n        generated_tokens = 0\n\n    combined_candidates = (existing + new_cands)[:target_calls]\n    voted_answer = compute_majority_vote([c.get("answer") for c in combined_candidates])\n    elapsed = time.perf_counter() - t0\n\n    return {\n        "candidates": combined_candidates,\n        "answer": voted_answer,\n        "generation_calls": generation_calls,\n        "total_samples": len(combined_candidates),\n        "generated_tokens": generated_tokens,\n        "time_seconds": float(elapsed),\n    }\n',
    "src/utils.py": '"""\nsrc/utils.py - Model loading, generation constants, seed management, and factories.\n"""\nimport json\nimport math\nimport os\nfrom typing import Any, Callable\n\n# Global random seed specified by course policy\nGLOBAL_SEED = 5326\n\n# Sampling & decoding hyperparameter configurations\nGREEDY_GEN_KWARGS = {\n    "do_sample": False,\n    "temperature": None,\n    "top_p": None,\n    "top_k": None,\n    "max_new_tokens": 512,\n}\n\nSAMPLING_GEN_KWARGS = {\n    "do_sample": True,\n    "temperature": 0.7,\n    "top_p": 1.0,\n    "top_k": 0,\n    "repetition_penalty": 1.0,\n    "max_new_tokens": 512,\n}\n\nJUDGE_GEN_KWARGS = {\n    "do_sample": False,\n    "temperature": None,\n    "top_p": None,\n    "top_k": None,\n    "max_new_tokens": 96,\n}\n\n\ndef set_seed_for_problem(problem_index: int) -> None:\n    """\n    Sets a deterministic seed for problem_index (5326 + problem_index).\n    Guarantees reproducibility and safe resumability after interruptions.\n    """\n    try:\n        import transformers\n        transformers.set_seed(GLOBAL_SEED + problem_index)\n    except ImportError:\n        import random\n        random.seed(GLOBAL_SEED + problem_index)\n\n\ndef load_models(generator_device: str = "cuda:0", judge_device: str = "cuda:1"):\n    """\n    Loads Qwen2.5-1.5B-Instruct in float16 (Generator) and\n    Qwen2.5-7B-Instruct in 4-bit NF4 (Judge).\n    \n    Hardware Mapping:\n      - Kaggle GPU T4x2: generator on cuda:0, judge on cuda:1.\n      - Single GPU: both models placed on cuda:0.\n      - Warning: Never use device_map="auto" across two GPUs for a single model!\n    """\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0\n    if num_gpus < 2:\n        g_dev = "cuda:0" if num_gpus == 1 else "cpu"\n        j_dev = "cuda:0" if num_gpus == 1 else "cpu"\n    else:\n        g_dev = generator_device\n        j_dev = judge_device\n\n    # 1. Generator: Qwen/Qwen2.5-1.5B-Instruct in float16 (~3.1 GB VRAM)\n    gen_id = "Qwen/Qwen2.5-1.5B-Instruct"\n    gen_tokenizer = AutoTokenizer.from_pretrained(gen_id)\n    if gen_tokenizer.pad_token_id is None:\n        gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id\n\n    gen_device_map = {"": g_dev} if g_dev != "cpu" else None\n    gen_model = AutoModelForCausalLM.from_pretrained(\n        gen_id,\n        torch_dtype=torch.float16 if g_dev != "cpu" else torch.float32,\n        device_map=gen_device_map,\n    )\n    if g_dev == "cpu":\n        gen_model = gen_model.to("cpu")\n\n    # 2. Judge: Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 (~5.5 GB VRAM)\n    judge_id = "Qwen/Qwen2.5-7B-Instruct"\n    judge_tokenizer = AutoTokenizer.from_pretrained(judge_id)\n    if judge_tokenizer.pad_token_id is None:\n        judge_tokenizer.pad_token_id = judge_tokenizer.eos_token_id\n\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_use_double_quant=True,\n    )\n    judge_device_map = {"": j_dev} if j_dev != "cpu" else None\n    judge_model = AutoModelForCausalLM.from_pretrained(\n        judge_id,\n        quantization_config=bnb_config if j_dev != "cpu" else None,\n        torch_dtype=torch.float16 if j_dev != "cpu" else torch.float32,\n        device_map=judge_device_map,\n    )\n    if j_dev == "cpu":\n        judge_model = judge_model.to("cpu")\n\n    return gen_model, gen_tokenizer, judge_model, judge_tokenizer\n\n\ndef _get_eos_token_ids(model: Any, tokenizer: Any) -> set[int]:\n    """Extracts all EOS token IDs from model.generation_config and tokenizer."""\n    eos_ids: set[int] = set()\n    if hasattr(model, "generation_config") and model.generation_config is not None:\n        cfg_eos = getattr(model.generation_config, "eos_token_id", None)\n        if cfg_eos is not None:\n            if isinstance(cfg_eos, int):\n                eos_ids.add(cfg_eos)\n            elif isinstance(cfg_eos, (list, tuple, set)):\n                eos_ids.update(cfg_eos)\n    if tokenizer is not None:\n        tok_eos = getattr(tokenizer, "eos_token_id", None)\n        if tok_eos is not None:\n            if isinstance(tok_eos, int):\n                eos_ids.add(tok_eos)\n            elif isinstance(tok_eos, (list, tuple, set)):\n                eos_ids.update(tok_eos)\n    return eos_ids\n\n\ndef _slice_to_first_eos(gen_tokens: Any, eos_ids: set[int]) -> Any:\n    """Slices token sequence up to and including the first EOS token."""\n    if hasattr(gen_tokens, "tolist"):\n        ids = gen_tokens.tolist()\n    else:\n        ids = list(gen_tokens)\n    first_eos_idx = None\n    for idx, tok in enumerate(ids):\n        if tok in eos_ids:\n            first_eos_idx = idx\n            break\n    if first_eos_idx is not None:\n        return gen_tokens[:first_eos_idx + 1]\n    return gen_tokens\n\n\ndef _extract_verdict_logprob(\n    gen_tokens_valid: Any,\n    step_scores_tuple: Any,\n    valid_token_pairs: list[tuple[int, int]],\n) -> float | None:\n    """\n    Computes continuous step validity probability p_t from the judge\'s raw logits\n    at the verdict token:\n        p_t = exp(z_VALID) / (exp(z_VALID) + exp(z_INVALID))\n            = 1 / (1 + exp(z_INVALID - z_VALID))\n    """\n    if not step_scores_tuple or not valid_token_pairs:\n        return None\n\n    try:\n        token_ids = gen_tokens_valid.tolist() if hasattr(gen_tokens_valid, "tolist") else list(gen_tokens_valid)\n        target_step_idx = None\n        matched_pair = valid_token_pairs[0]\n\n        all_valid_ids = {p[0] for p in valid_token_pairs}\n        all_invalid_ids = {p[1] for p in valid_token_pairs}\n\n        for idx, tok_id in enumerate(token_ids):\n            if idx >= len(step_scores_tuple):\n                break\n            if tok_id in all_valid_ids or tok_id in all_invalid_ids:\n                target_step_idx = idx\n                for v_id, inv_id in valid_token_pairs:\n                    if tok_id == v_id or tok_id == inv_id:\n                        matched_pair = (v_id, inv_id)\n                        break\n                break\n\n        if target_step_idx is None:\n            return None\n\n        logits_step = step_scores_tuple[target_step_idx][0]\n        v_id, inv_id = matched_pair\n        z_valid = float(logits_step[v_id].item() if hasattr(logits_step[v_id], "item") else logits_step[v_id])\n        z_invalid = float(logits_step[inv_id].item() if hasattr(logits_step[inv_id], "item") else logits_step[inv_id])\n\n        diff = max(-50.0, min(50.0, z_valid - z_invalid))\n        prob = 1.0 / (1.0 + math.exp(-diff))\n        return max(1e-4, min(0.9999, prob))\n    except Exception:\n        return None\n\n\ndef make_hf_generate_fn(model: Any, tokenizer: Any) -> Callable[[str, int, dict], list[dict]]:\n    """\n    Factory creating a GenerateFn conforming to submission_adapter contract.\n    Applies the Qwen chat template to raw instruction prompts.\n    """\n    def generate_fn(prompt: str, num_samples: int, gen_kwargs: dict) -> list[dict]:\n        try:\n            import torch\n            no_grad_ctx = torch.no_grad\n        except ImportError:\n            class _NoGrad:\n                def __enter__(self): return self\n                def __exit__(self, *args): pass\n            no_grad_ctx = lambda: _NoGrad()\n\n        messages = [{"role": "user", "content": prompt}]\n        formatted_prompt = tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True\n        )\n        inputs = tokenizer(formatted_prompt, return_tensors="pt")\n        inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}\n        prompt_len = inputs["input_ids"].shape[1]\n\n        kwargs = dict(gen_kwargs)\n        if not kwargs.get("do_sample", False):\n            kwargs["temperature"] = None\n            kwargs["top_p"] = None\n            kwargs["top_k"] = None\n\n        if num_samples > 1 and kwargs.get("do_sample", True):\n            kwargs["num_return_sequences"] = num_samples\n\n        kwargs.setdefault("pad_token_id", tokenizer.eos_token_id)\n\n        with no_grad_ctx():\n            output_ids = model.generate(**inputs, **kwargs)\n\n        eos_ids = _get_eos_token_ids(model, tokenizer)\n        results = []\n        for i in range(output_ids.shape[0]):\n            gen_tokens = output_ids[i][prompt_len:]\n            gen_tokens_valid = _slice_to_first_eos(gen_tokens, eos_ids)\n            text = tokenizer.decode(gen_tokens_valid, skip_special_tokens=True)\n            results.append({"text": text, "num_tokens": len(gen_tokens_valid)})\n        return results\n\n    return generate_fn\n\n\ndef make_hf_judge_fn(judge_model: Any, judge_tokenizer: Any) -> Callable[[str], dict]:\n    """\n    Factory creating a JudgeFn conforming to submission_adapter contract.\n    Applies the Qwen chat template to raw judge prompts.\n    When used with a full HuggingFace tokenizer/model, also extracts continuous\n    step confidence \'valid_prob\' from the verdict token\'s logits (Optional Bonus).\n    """\n    valid_token_pairs: list[tuple[int, int]] = []\n    supports_encode = hasattr(judge_tokenizer, "encode")\n    if supports_encode:\n        try:\n            for v_str, inv_str in [(" VALID", " INVALID"), ("VALID", "INVALID"), (" valid", " invalid")]:\n                v_ids = judge_tokenizer.encode(v_str, add_special_tokens=False)\n                inv_ids = judge_tokenizer.encode(inv_str, add_special_tokens=False)\n                if v_ids and inv_ids:\n                    pair = (int(v_ids[0]), int(inv_ids[0]))\n                    if pair not in valid_token_pairs:\n                        valid_token_pairs.append(pair)\n        except Exception:\n            valid_token_pairs = []\n\n    def judge_fn(judge_prompt: str) -> dict:\n        try:\n            import torch\n            no_grad_ctx = torch.no_grad\n        except ImportError:\n            class _NoGrad:\n                def __enter__(self): return self\n                def __exit__(self, *args): pass\n            no_grad_ctx = lambda: _NoGrad()\n\n        messages = [{"role": "user", "content": judge_prompt}]\n        formatted = judge_tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True\n        )\n        inputs = judge_tokenizer(formatted, return_tensors="pt")\n        inputs = {k: v.to(judge_model.device) if hasattr(v, "to") else v for k, v in inputs.items()}\n        prompt_len = inputs["input_ids"].shape[1]\n\n        kwargs = dict(JUDGE_GEN_KWARGS)\n        kwargs.setdefault("pad_token_id", judge_tokenizer.eos_token_id)\n        if valid_token_pairs:\n            kwargs["output_scores"] = True\n            kwargs["return_dict_in_generate"] = True\n\n        with no_grad_ctx():\n            raw_output = judge_model.generate(**inputs, **kwargs)\n\n        if hasattr(raw_output, "sequences"):\n            output_ids = raw_output.sequences\n            step_scores_tuple = getattr(raw_output, "scores", None)\n        else:\n            output_ids = raw_output\n            step_scores_tuple = None\n\n        eos_ids = _get_eos_token_ids(judge_model, judge_tokenizer)\n        gen_tokens = output_ids[0][prompt_len:]\n        gen_tokens_valid = _slice_to_first_eos(gen_tokens, eos_ids)\n        text = judge_tokenizer.decode(gen_tokens_valid, skip_special_tokens=True)\n\n        result: dict[str, Any] = {"text": text, "num_tokens": len(gen_tokens_valid)}\n        if step_scores_tuple is not None and valid_token_pairs:\n            v_prob = _extract_verdict_logprob(gen_tokens_valid, step_scores_tuple, valid_token_pairs)\n            if v_prob is not None:\n                result["valid_prob"] = float(v_prob)\n        return result\n\n    return judge_fn\n\n\ndef save_results_summary(results_list: list[dict[str, Any]], filepath: str = "outputs/results_summary.json") -> None:\n    """\n    Saves results_summary.json conforming strictly to the assignment schema:\n    Keys: \'Method\', \'Accuracy (%)\', \'Generation Calls\', \'Judge Calls\', \'Total LLM Calls\',\n          \'Avg Gen Tokens / Problem\', \'Avg Judge Tokens / Problem\', \'Total Time (s)\'\n    """\n    os.makedirs(os.path.dirname(filepath), exist_ok=True)\n    with open(filepath, "w", encoding="utf-8") as f:\n        json.dump(results_list, f, indent=2)\n\n\ndef save_aggregation_ablation(ablation_list: list[dict[str, Any]], filepath: str = "outputs/aggregation_ablation.json") -> None:\n    """\n    Saves aggregation_ablation.json:\n    List of {\'Strategy\': str, \'Lambda\': float | None, \'Accuracy (%)\': float, \'Tie Rate (%)\': float}\n    """\n    os.makedirs(os.path.dirname(filepath), exist_ok=True)\n    with open(filepath, "w", encoding="utf-8") as f:\n        json.dump(ablation_list, f, indent=2)\n\n\ndef save_sc_scaling_curve(curve_list: list[dict[str, Any]], filepath: str = "outputs/sc_scaling_curve.json") -> None:\n    """\n    Saves sc_scaling_curve.json:\n    List of {\'N\': int, \'Accuracy (%)\': float, \'Avg Gen Tokens / Problem\': float}\n    """\n    os.makedirs(os.path.dirname(filepath), exist_ok=True)\n    with open(filepath, "w", encoding="utf-8") as f:\n        json.dump(curve_list, f, indent=2)\n',
    "run_all.py": '"""\nrun_all.py - End-to-end evaluation pipeline for CS 5326 Programming Assignment 3.\n\nFeatures:\n  - Evaluates all 6 required configurations + lambda sweep + optional bonus (product_logprob)\n    across the 50-problem GSM8K benchmark (data/gsm8k_test_50.jsonl).\n  - Enforces per-problem deterministic seeding: set_seed_for_problem(problem_index) -> 5326 + i.\n  - Shares the 5 sampled paths across Self-Consistency (N=5) and all PRM Best-of-5 variants,\n    scoring steps once with the 7B judge and reusing step scores across aggregation rules.\n  - Extends the 5 shared paths with 10 additional samples for Compute-Matched SC (N=15)\n    and computes the full Self-Consistency scaling curve for N = 1..15.\n  - Incrementally checkpoints every completed problem to outputs/per_problem.jsonl so\n    interrupted Kaggle sessions resume seamlessly without repeating work.\n  - Automatically computes and saves:\n      * outputs/results_summary.json\n      * outputs/aggregation_ablation.json\n      * outputs/sc_scaling_curve.json\n      * outputs/per_problem.jsonl\n\nUsage on Kaggle (GPU T4x2):\n  python run_all.py\n"""\nimport argparse\nimport copy\nimport json\nimport os\nimport time\nfrom typing import Any\n\nfrom src.extraction import extract_answer, parse_steps\nfrom src.generation import (\n    compute_majority_vote,\n    format_step_prompt,\n    run_greedy,\n    run_self_consistency,\n)\nfrom src.prm_judge import score_path\nfrom src.search import (\n    aggregate_path_score,\n    run_compute_matched_self_consistency,\n    select_best_path,\n)\nfrom src.utils import (\n    GLOBAL_SEED,\n    load_models,\n    make_hf_generate_fn,\n    make_hf_judge_fn,\n    save_aggregation_ablation,\n    save_results_summary,\n    save_sc_scaling_curve,\n    set_seed_for_problem,\n)\n\n\ndef load_benchmark(dataset_path: str = "data/gsm8k_test_50.jsonl") -> list[dict[str, Any]]:\n    """Loads the 50-problem GSM8K evaluation benchmark."""\n    problems: list[dict[str, Any]] = []\n    with open(dataset_path, "r", encoding="utf-8") as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                problems.append(json.loads(line))\n    return problems\n\n\ndef load_existing_checkpoints(per_problem_path: str) -> dict[str, dict[str, Any]]:\n    """Loads already-evaluated problem records from outputs/per_problem.jsonl for resumption."""\n    completed: dict[str, dict[str, Any]] = {}\n    if not os.path.exists(per_problem_path):\n        return completed\n    with open(per_problem_path, "r", encoding="utf-8") as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                rec = json.loads(line)\n                if "id" in rec and "candidates" in rec and len(rec["candidates"]) == 15:\n                    completed[rec["id"]] = rec\n            except Exception:\n                continue\n    return completed\n\n\ndef write_all_checkpoints(records: list[dict[str, Any]], per_problem_path: str) -> None:\n    """Writes all completed problem records in canonical order to outputs/per_problem.jsonl."""\n    os.makedirs(os.path.dirname(per_problem_path), exist_ok=True)\n    tmp_path = per_problem_path + ".tmp"\n    with open(tmp_path, "w", encoding="utf-8") as f:\n        for rec in records:\n            f.write(json.dumps(rec) + "\\n")\n    os.replace(tmp_path, per_problem_path)\n\n\ndef evaluate_single_problem(\n    problem_index: int,\n    problem: dict[str, Any],\n    gen_fn: Any,\n    judge_fn: Any,\n) -> dict[str, Any]:\n    """\n    Runs all configurations on a single benchmark problem with shared candidate pool:\n      1. Greedy (N=1)\n      2. Self-Consistency (N=5)\n      3. PRM Judge over the 5 shared candidates (cached across aggregations)\n      4. PRM Best-of-5 across min, product, step_penalty (lambda in {0.0, 0.5, 2.0}), and product_logprob\n      5. Compute-Matched Self-Consistency (N=15, sampling 10 additional candidates)\n    """\n    set_seed_for_problem(problem_index)\n    prob_id = problem["id"]\n    question = problem["question"]\n    gt_answer = int(problem["ground_truth_answer"])\n\n    # 1. Greedy Baseline (N=1)\n    greedy_res = run_greedy(question=question, generate_fn=gen_fn)\n\n    # 2. Self-Consistency (N=5) - Shared 5-Candidate Pool\n    sc5_res = run_self_consistency(\n        question=question,\n        n=5,\n        temperature=0.7,\n        generate_fn=gen_fn,\n    )\n    shared_5 = [copy.deepcopy(c) for c in sc5_res["candidates"]]\n\n    # 3. PRM Step-Level Verification over the 5 shared candidates (scored once, reused across rules)\n    t_judge_start = time.perf_counter()\n    judge_calls_total = 0\n    judge_tokens_total = 0\n    scored_5: list[dict[str, Any]] = []\n\n    for cand in shared_5:\n        cand_copy = copy.deepcopy(cand)\n        s_res = score_path(\n            question=question,\n            candidate=cand_copy,\n            judge_fn=judge_fn,\n        )\n        cand_copy["step_scores"] = s_res["step_scores"]\n        cand_copy["rationales"] = s_res["rationales"]\n        if "step_probs" in s_res:\n            cand_copy["step_probs"] = s_res["step_probs"]\n        judge_calls_total += int(s_res["judge_calls"])\n        judge_tokens_total += int(s_res["judge_tokens"])\n        scored_5.append(cand_copy)\n\n    judge_time_s = float(time.perf_counter() - t_judge_start)\n\n    # 4. PRM Best-of-5 Selections across aggregation strategies\n    prm_selections: dict[str, dict[str, Any]] = {}\n    agg_configs = [\n        ("min", "min", 2.0),\n        ("product", "product", 2.0),\n        ("step_penalty_0.0", "step_penalty", 0.0),\n        ("step_penalty_0.5", "step_penalty", 0.5),\n        ("step_penalty_2.0", "step_penalty", 2.0),\n        ("product_logprob", "product_logprob", 2.0),\n    ]\n    for key_name, strat, lam in agg_configs:\n        t_sel = time.perf_counter()\n        sel = select_best_path(scored_5, aggregation_strategy=strat, penalty_lambda=lam)\n        sel_time = float(time.perf_counter() - t_sel)\n        prm_selections[key_name] = {\n            "answer": sel.get("answer"),\n            "path_score": float(sel["path_score"]),\n            "num_tied": int(sel["num_tied"]),\n            "time_seconds": sel_time,\n        }\n\n    # 5. Compute-Matched Self-Consistency (N=15): retains scored_5 (indices 0..4) + samples 10 new (indices 5..14)\n    sc15_res = run_compute_matched_self_consistency(\n        question=question,\n        target_calls=15,\n        temperature=0.7,\n        existing_candidates=scored_5,\n        generate_fn=gen_fn,\n    )\n    all_15_candidates = sc15_res["candidates"]\n\n    # Scaling curve answers for N = 1..15\n    sc_curve_answers = [\n        compute_majority_vote([c.get("answer") for c in all_15_candidates[:n_sub]])\n        for n_sub in range(1, 16)\n    ]\n    sc_curve_tokens = [\n        sum(int(c.get("num_tokens", 0)) for c in all_15_candidates[:n_sub])\n        for n_sub in range(1, 16)\n    ]\n\n    record = {\n        "id": prob_id,\n        "problem_index": problem_index,\n        "question": question,\n        "ground_truth_answer": gt_answer,\n        "greedy": {\n            "completion": greedy_res["completion"],\n            "steps": greedy_res["steps"],\n            "answer": greedy_res["answer"],\n            "correct": greedy_res["answer"] == gt_answer,\n            "generation_calls": int(greedy_res["generation_calls"]),\n            "generated_tokens": int(greedy_res["generated_tokens"]),\n            "time_seconds": float(greedy_res["time_seconds"]),\n        },\n        "shared_candidates_5": scored_5,\n        "candidates": all_15_candidates,\n        "predictions": {\n            "greedy": greedy_res["answer"],\n            "sc_5": sc5_res["answer"],\n            "sc_15": sc15_res["answer"],\n            "prm_min": prm_selections["min"]["answer"],\n            "prm_product": prm_selections["product"]["answer"],\n            "prm_step_penalty_0.0": prm_selections["step_penalty_0.0"]["answer"],\n            "prm_step_penalty_0.5": prm_selections["step_penalty_0.5"]["answer"],\n            "prm_step_penalty_2.0": prm_selections["step_penalty_2.0"]["answer"],\n            "prm_product_logprob": prm_selections["product_logprob"]["answer"],\n        },\n        "prm_selections": prm_selections,\n        "sc_curve_answers": sc_curve_answers,\n        "sc_curve_tokens": sc_curve_tokens,\n        "bookkeeping": {\n            "greedy_calls": int(greedy_res["generation_calls"]),\n            "greedy_tokens": int(greedy_res["generated_tokens"]),\n            "greedy_time_s": float(greedy_res["time_seconds"]),\n            "sc5_calls": int(sc5_res["generation_calls"]),\n            "sc5_tokens": int(sc5_res["generated_tokens"]),\n            "sc5_time_s": float(sc5_res["time_seconds"]),\n            "sc10_extra_calls": int(sc15_res["generation_calls"]),\n            "sc10_extra_tokens": int(sc15_res["generated_tokens"]),\n            "sc10_extra_time_s": float(sc15_res["time_seconds"]),\n            "sc15_calls": int(sc5_res["generation_calls"]) + int(sc15_res["generation_calls"]),\n            "sc15_tokens": int(sc5_res["generated_tokens"]) + int(sc15_res["generated_tokens"]),\n            "sc15_time_s": float(sc5_res["time_seconds"]) + float(sc15_res["time_seconds"]),\n            "judge_calls": judge_calls_total,\n            "judge_tokens": judge_tokens_total,\n            "judge_time_s": judge_time_s,\n        },\n    }\n    return record\n\n\ndef compute_and_save_artifacts(\n    records: list[dict[str, Any]],\n    output_dir: str = "outputs",\n) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:\n    """\n    Computes and saves results_summary.json, aggregation_ablation.json, and sc_scaling_curve.json\n    from the per-problem evaluation records.\n    """\n    n_probs = len(records)\n    if n_probs == 0:\n        raise ValueError("No problem records available to summarize.")\n\n    def acc_pct(pred_key: str) -> float:\n        correct = sum(\n            1 for r in records if r["predictions"].get(pred_key) == r["ground_truth_answer"]\n        )\n        return round(100.0 * correct / n_probs, 2)\n\n    # Aggregate call, token, and time totals across all 50 problems\n    greedy_gen_calls = sum(r["bookkeeping"]["greedy_calls"] for r in records)\n    greedy_gen_tokens = sum(r["bookkeeping"]["greedy_tokens"] for r in records)\n    greedy_time = sum(r["bookkeeping"]["greedy_time_s"] for r in records)\n\n    sc5_gen_calls = sum(r["bookkeeping"]["sc5_calls"] for r in records)\n    sc5_gen_tokens = sum(r["bookkeeping"]["sc5_tokens"] for r in records)\n    sc5_time = sum(r["bookkeeping"]["sc5_time_s"] for r in records)\n\n    sc15_gen_calls = sum(r["bookkeeping"]["sc15_calls"] for r in records)\n    sc15_gen_tokens = sum(r["bookkeeping"]["sc15_tokens"] for r in records)\n    sc15_time = sum(r["bookkeeping"]["sc15_time_s"] for r in records)\n\n    judge_calls = sum(r["bookkeeping"]["judge_calls"] for r in records)\n    judge_tokens = sum(r["bookkeeping"]["judge_tokens"] for r in records)\n    judge_time = sum(r["bookkeeping"]["judge_time_s"] for r in records)\n\n    prm_min_sel_time = sum(r["prm_selections"]["min"]["time_seconds"] for r in records)\n    prm_prod_sel_time = sum(r["prm_selections"]["product"]["time_seconds"] for r in records)\n    prm_pen2_sel_time = sum(r["prm_selections"]["step_penalty_2.0"]["time_seconds"] for r in records)\n\n    # 1. results_summary.json (6 mandatory rows)\n    results_summary = [\n        {\n            "Method": "Greedy (N=1)",\n            "Accuracy (%)": acc_pct("greedy"),\n            "Generation Calls": greedy_gen_calls,\n            "Judge Calls": 0,\n            "Total LLM Calls": greedy_gen_calls,\n            "Avg Gen Tokens / Problem": round(greedy_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": 0.0,\n            "Total Time (s)": round(greedy_time, 2),\n        },\n        {\n            "Method": "Self-Consistency (N=5)",\n            "Accuracy (%)": acc_pct("sc_5"),\n            "Generation Calls": sc5_gen_calls,\n            "Judge Calls": 0,\n            "Total LLM Calls": sc5_gen_calls,\n            "Avg Gen Tokens / Problem": round(sc5_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": 0.0,\n            "Total Time (s)": round(sc5_time, 2),\n        },\n        {\n            "Method": "Compute-Matched SC (N=15)",\n            "Accuracy (%)": acc_pct("sc_15"),\n            "Generation Calls": sc15_gen_calls,\n            "Judge Calls": 0,\n            "Total LLM Calls": sc15_gen_calls,\n            "Avg Gen Tokens / Problem": round(sc15_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": 0.0,\n            "Total Time (s)": round(sc15_time, 2),\n        },\n        {\n            "Method": "PRM Best-of-5 (min)",\n            "Accuracy (%)": acc_pct("prm_min"),\n            "Generation Calls": sc5_gen_calls,\n            "Judge Calls": judge_calls,\n            "Total LLM Calls": sc5_gen_calls + judge_calls,\n            "Avg Gen Tokens / Problem": round(sc5_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": round(judge_tokens / n_probs, 2),\n            "Total Time (s)": round(sc5_time + judge_time + prm_min_sel_time, 2),\n        },\n        {\n            "Method": "PRM Best-of-5 (product)",\n            "Accuracy (%)": acc_pct("prm_product"),\n            "Generation Calls": sc5_gen_calls,\n            "Judge Calls": judge_calls,\n            "Total LLM Calls": sc5_gen_calls + judge_calls,\n            "Avg Gen Tokens / Problem": round(sc5_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": round(judge_tokens / n_probs, 2),\n            "Total Time (s)": round(sc5_time + judge_time + prm_prod_sel_time, 2),\n        },\n        {\n            "Method": "PRM Best-of-5 (step_penalty, lambda=2.0)",\n            "Accuracy (%)": acc_pct("prm_step_penalty_2.0"),\n            "Generation Calls": sc5_gen_calls,\n            "Judge Calls": judge_calls,\n            "Total LLM Calls": sc5_gen_calls + judge_calls,\n            "Avg Gen Tokens / Problem": round(sc5_gen_tokens / n_probs, 2),\n            "Avg Judge Tokens / Problem": round(judge_tokens / n_probs, 2),\n            "Total Time (s)": round(sc5_time + judge_time + prm_pen2_sel_time, 2),\n        },\n    ]\n\n    # 2. aggregation_ablation.json\n    def tie_rate_pct(sel_key: str) -> float:\n        ties = sum(1 for r in records if int(r["prm_selections"][sel_key]["num_tied"]) > 1)\n        return round(100.0 * ties / n_probs, 2)\n\n    aggregation_ablation = [\n        {\n            "Strategy": "min",\n            "Lambda": None,\n            "Accuracy (%)": acc_pct("prm_min"),\n            "Tie Rate (%)": tie_rate_pct("min"),\n        },\n        {\n            "Strategy": "product",\n            "Lambda": None,\n            "Accuracy (%)": acc_pct("prm_product"),\n            "Tie Rate (%)": tie_rate_pct("product"),\n        },\n        {\n            "Strategy": "step_penalty",\n            "Lambda": 0.0,\n            "Accuracy (%)": acc_pct("prm_step_penalty_0.0"),\n            "Tie Rate (%)": tie_rate_pct("step_penalty_0.0"),\n        },\n        {\n            "Strategy": "step_penalty",\n            "Lambda": 0.5,\n            "Accuracy (%)": acc_pct("prm_step_penalty_0.5"),\n            "Tie Rate (%)": tie_rate_pct("step_penalty_0.5"),\n        },\n        {\n            "Strategy": "step_penalty",\n            "Lambda": 2.0,\n            "Accuracy (%)": acc_pct("prm_step_penalty_2.0"),\n            "Tie Rate (%)": tie_rate_pct("step_penalty_2.0"),\n        },\n    ]\n\n    # Save bonus product_logprob stats alongside in outputs/bonus_product_logprob.json\n    bonus_stats = {\n        "Strategy": "product_logprob",\n        "Lambda": None,\n        "Accuracy (%)": acc_pct("prm_product_logprob"),\n        "Tie Rate (%)": tie_rate_pct("product_logprob"),\n    }\n    os.makedirs(output_dir, exist_ok=True)\n    with open(os.path.join(output_dir, "bonus_product_logprob.json"), "w", encoding="utf-8") as f:\n        json.dump(bonus_stats, f, indent=2)\n\n    # 3. sc_scaling_curve.json (N = 1..15)\n    sc_scaling_curve = []\n    for n_sub in range(1, 16):\n        idx = n_sub - 1\n        correct_n = sum(\n            1\n            for r in records\n            if compute_majority_vote([c.get("answer") for c in r["candidates"][:n_sub]])\n            == r["ground_truth_answer"]\n        )\n        tokens_n = sum(\n            sum(int(c.get("num_tokens", 0)) for c in r["candidates"][:n_sub])\n            for r in records\n        )\n        sc_scaling_curve.append({\n            "N": n_sub,\n            "Accuracy (%)": round(100.0 * correct_n / n_probs, 2),\n            "Avg Gen Tokens / Problem": round(tokens_n / n_probs, 2),\n        })\n\n    save_results_summary(results_summary, os.path.join(output_dir, "results_summary.json"))\n    save_aggregation_ablation(aggregation_ablation, os.path.join(output_dir, "aggregation_ablation.json"))\n    save_sc_scaling_curve(sc_scaling_curve, os.path.join(output_dir, "sc_scaling_curve.json"))\n\n    return results_summary, aggregation_ablation, sc_scaling_curve\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run CS 5326 PA3 GSM8K Benchmark Evaluation")\n    parser.add_argument("--dataset", type=str, default="data/gsm8k_test_50.jsonl")\n    parser.add_argument("--output-dir", type=str, default="outputs")\n    parser.add_argument("--gen-device", type=str, default="cuda:0")\n    parser.add_argument("--judge-device", type=str, default="cuda:1")\n    parser.add_argument("--mock", action="store_true", help="Run offline with synthetic mock callables for pipeline testing")\n    args = parser.parse_args()\n\n    problems = load_benchmark(args.dataset)\n    per_problem_path = os.path.join(args.output_dir, "per_problem.jsonl")\n    completed_map = load_existing_checkpoints(per_problem_path)\n    print(f"Loaded {len(problems)} problems from {args.dataset}. Found {len(completed_map)} existing checkpoints.")\n\n    if args.mock:\n        print("Running in offline --mock mode...")\n        def mock_gen_fn(prompt: str, num_samples: int, gen_kwargs: dict) -> list[dict]:\n            return [\n                {\n                    "text": "Step 1: Compute intermediate value 9 * 2 = 18.\\nFinal Answer: 18",\n                    "num_tokens": 16,\n                }\n                for _ in range(num_samples)\n            ]\n\n        def mock_judge_fn(judge_prompt: str) -> dict:\n            return {\n                "text": "Verdict: VALID\\nRationale: The arithmetic and logic are sound.",\n                "num_tokens": 12,\n                "valid_prob": 0.96,\n            }\n\n        gen_fn = mock_gen_fn\n        judge_fn = mock_judge_fn\n    else:\n        if len(completed_map) < len(problems):\n            print(f"Loading Generator on {args.gen_device} and PRM Judge on {args.judge_device}...")\n            gen_model, gen_tokenizer, judge_model, judge_tokenizer = load_models(\n                generator_device=args.gen_device,\n                judge_device=args.judge_device,\n            )\n            gen_fn = make_hf_generate_fn(gen_model, gen_tokenizer)\n            judge_fn = make_hf_judge_fn(judge_model, judge_tokenizer)\n        else:\n            gen_fn = None\n            judge_fn = None\n\n    ordered_records: list[dict[str, Any]] = []\n    for idx, prob in enumerate(problems):\n        prob_id = prob["id"]\n        if prob_id in completed_map:\n            rec = completed_map[prob_id]\n            ordered_records.append(rec)\n            print(f"[{idx + 1:02d}/{len(problems):02d}] {prob_id} (restored from checkpoint)")\n            continue\n\n        t0 = time.perf_counter()\n        rec = evaluate_single_problem(\n            problem_index=idx,\n            problem=prob,\n            gen_fn=gen_fn,\n            judge_fn=judge_fn,\n        )\n        ordered_records.append(rec)\n        completed_map[prob_id] = rec\n\n        # Save incremental checkpoint immediately\n        write_all_checkpoints(ordered_records, per_problem_path)\n        dt = time.perf_counter() - t0\n        preds = rec["predictions"]\n        gt = rec["ground_truth_answer"]\n        print(\n            f"[{idx + 1:02d}/{len(problems):02d}] {prob_id} | GT={gt} | "\n            f"Greedy={preds[\'greedy\']} | SC@5={preds[\'sc_5\']} | "\n            f"PRM(min)={preds[\'prm_min\']} | SC@15={preds[\'sc_15\']} | ({dt:.1f}s)"\n        )\n\n    # Ensure final canonical order in per_problem.jsonl\n    write_all_checkpoints(ordered_records, per_problem_path)\n    summary, ablation, curve = compute_and_save_artifacts(ordered_records, args.output_dir)\n\n    print("\\n=== Benchmark Summary (outputs/results_summary.json) ===")\n    for row in summary:\n        print(\n            f"  {row[\'Method\']:<40} | Acc: {row[\'Accuracy (%)\']:>6.2f}% | "\n            f"Calls: {row[\'Total LLM Calls\']:>4d} | Time: {row[\'Total Time (s)\']:>7.2f}s"\n        )\n\n    print("\\n=== Aggregation Ablation (outputs/aggregation_ablation.json) ===")\n    for row in ablation:\n        print(\n            f"  Strategy: {row[\'Strategy\']:<15} | Lambda: {str(row[\'Lambda\']):<5} | "\n            f"Acc: {row[\'Accuracy (%)\']:>6.2f}% | Tie Rate: {row[\'Tie Rate (%)\']:>6.2f}%"\n        )\n\n\nif __name__ == "__main__":\n    main()\n\n',
    "analyze_outputs.py": '"""\nanalyze_outputs.py - Statistical analysis, bootstrap CIs, failure case mining,\njudge audit extraction, and publication-quality plot generation for REPORT.pdf.\n\nUsage:\n  python3 analyze_outputs.py --output-dir outputs --fig-dir figures\n"""\nimport argparse\nimport json\nimport os\nimport random\nfrom typing import Any\n\nfrom src.generation import compute_majority_vote\n\n\ndef load_per_problem(path: str) -> list[dict[str, Any]]:\n    records = []\n    with open(path, "r", encoding="utf-8") as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                records.append(json.loads(line))\n    return records\n\n\ndef compute_bootstrap_ci(\n    correct_flags: list[int],\n    n_resamples: int = 1000,\n    seed: int = 5326,\n) -> tuple[float, float, float]:\n    """\n    Computes mean accuracy (%) and 95% bootstrap confidence interval (2.5th, 97.5th percentiles)\n    using 1,000 resamples over problems with seed 5326.\n    """\n    rng = random.Random(seed)\n    n = len(correct_flags)\n    if n == 0:\n        return 0.0, 0.0, 0.0\n    mean_acc = 100.0 * sum(correct_flags) / n\n    boot_means = []\n    for _ in range(n_resamples):\n        sample = [correct_flags[rng.randrange(n)] for _ in range(n)]\n        boot_means.append(100.0 * sum(sample) / n)\n    boot_means.sort()\n    low_idx = int(0.025 * n_resamples)\n    high_idx = int(0.975 * n_resamples) - 1\n    return round(mean_acc, 2), round(boot_means[low_idx], 2), round(boot_means[high_idx], 2)\n\n\ndef compute_paired_win_loss(\n    records: list[dict[str, Any]],\n    key_a: str,\n    key_b: str,\n) -> dict[str, int]:\n    """\n    Compares method A vs method B on each problem against ground_truth_answer:\n      - a_wins: A is correct and B is wrong\n      - b_wins: B is correct and A is wrong\n      - both_correct: both A and B are correct\n      - both_wrong: both A and B are wrong\n    """\n    a_wins = 0\n    b_wins = 0\n    both_correct = 0\n    both_wrong = 0\n    for r in records:\n        gt = r["ground_truth_answer"]\n        a_ok = r["predictions"].get(key_a) == gt\n        b_ok = r["predictions"].get(key_b) == gt\n        if a_ok and not b_ok:\n            a_wins += 1\n        elif b_ok and not a_ok:\n            b_wins += 1\n        elif a_ok and b_ok:\n            both_correct += 1\n        else:\n            both_wrong += 1\n    return {\n        "a_wins": a_wins,\n        "b_wins": b_wins,\n        "both_correct": both_correct,\n        "both_wrong": both_wrong,\n        "both_agreed_correctness": both_correct + both_wrong,\n    }\n\n\ndef generate_plots(\n    results_summary: list[dict[str, Any]],\n    scaling_curve: list[dict[str, Any]],\n    ablation: list[dict[str, Any]],\n    fig_dir: str,\n) -> None:\n    """Generates PDF plots for Section 1 and Section 2 of REPORT.pdf."""\n    try:\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n    except ImportError:\n        print("matplotlib not installed; skipping PDF plot generation.")\n        return\n\n    os.makedirs(fig_dir, exist_ok=True)\n\n    # Extract SC curve arrays\n    ns = [row["N"] for row in scaling_curve]\n    sc_accs = [row["Accuracy (%)"] for row in scaling_curve]\n    sc_toks = [row["Avg Gen Tokens / Problem"] for row in scaling_curve]\n    sc_calls_total = [row["N"] * 50 for row in scaling_curve]\n\n    summary_by_name = {row["Method"]: row for row in results_summary}\n    greedy_row = summary_by_name["Greedy (N=1)"]\n    prm_min_row = summary_by_name["PRM Best-of-5 (min)"]\n    prm_prod_row = summary_by_name["PRM Best-of-5 (product)"]\n    prm_pen_row = summary_by_name["PRM Best-of-5 (step_penalty, lambda=2.0)"]\n\n    # Plot 1: Accuracy vs. Total LLM Calls\n    fig, ax = plt.subplots(figsize=(6.2, 4.0))\n    ax.plot(\n        sc_calls_total,\n        sc_accs,\n        marker="o",\n        linewidth=2.0,\n        markersize=5,\n        color="#1f77b4",\n        label="Self-Consistency ($N=1\\\\dots 15$)",\n    )\n    ax.scatter(\n        [greedy_row["Total LLM Calls"]],\n        [greedy_row["Accuracy (%)"]],\n        marker="D",\n        s=75,\n        color="#7f7f7f",\n        zorder=5,\n        label=f"Greedy ($N=1$): {greedy_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_min_row["Total LLM Calls"]],\n        [prm_min_row["Accuracy (%)"]],\n        marker="*",\n        s=150,\n        color="#d62728",\n        zorder=6,\n        label=f"PRM Best-of-5 (min): {prm_min_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_prod_row["Total LLM Calls"]],\n        [prm_prod_row["Accuracy (%)"]],\n        marker="^",\n        s=90,\n        color="#2ca02c",\n        zorder=5,\n        label=f"PRM Best-of-5 (product): {prm_prod_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_pen_row["Total LLM Calls"]],\n        [prm_pen_row["Accuracy (%)"]],\n        marker="s",\n        s=80,\n        color="#9467bd",\n        zorder=5,\n        label=f"PRM Best-of-5 (step_penalty): {prm_pen_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.set_xlabel("Total LLM Calls across 50 Problems (Gen + Judge)", fontsize=10)\n    ax.set_ylabel("Exact-Match Accuracy (%)", fontsize=10)\n    ax.set_title("Accuracy vs. Total LLM Calls on GSM8K-50", fontsize=11, fontweight="bold")\n    ax.grid(True, linestyle="--", alpha=0.4)\n    ax.legend(loc="lower right", fontsize=8.5, frameon=True)\n    fig.tight_layout()\n    fig.savefig(os.path.join(fig_dir, "scaling_calls.pdf"))\n    fig.savefig(os.path.join(fig_dir, "scaling_calls.png"), dpi=200)\n    plt.close(fig)\n\n    # Plot 2: Accuracy vs. Total Output Tokens per Problem\n    prm_total_tok_per_prob = (\n        prm_min_row["Avg Gen Tokens / Problem"] + prm_min_row["Avg Judge Tokens / Problem"]\n    )\n    fig, ax = plt.subplots(figsize=(6.2, 4.0))\n    ax.plot(\n        sc_toks,\n        sc_accs,\n        marker="o",\n        linewidth=2.0,\n        markersize=5,\n        color="#1f77b4",\n        label="Self-Consistency ($N=1\\\\dots 15$)",\n    )\n    ax.scatter(\n        [greedy_row["Avg Gen Tokens / Problem"]],\n        [greedy_row["Accuracy (%)"]],\n        marker="D",\n        s=75,\n        color="#7f7f7f",\n        zorder=5,\n        label=f"Greedy ($N=1$): {greedy_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_total_tok_per_prob],\n        [prm_min_row["Accuracy (%)"]],\n        marker="*",\n        s=150,\n        color="#d62728",\n        zorder=6,\n        label=f"PRM Best-of-5 (min): {prm_min_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_total_tok_per_prob],\n        [prm_prod_row["Accuracy (%)"]],\n        marker="^",\n        s=90,\n        color="#2ca02c",\n        zorder=5,\n        label=f"PRM Best-of-5 (product): {prm_prod_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.scatter(\n        [prm_total_tok_per_prob],\n        [prm_pen_row["Accuracy (%)"]],\n        marker="s",\n        s=80,\n        color="#9467bd",\n        zorder=5,\n        label=f"PRM Best-of-5 (step_penalty): {prm_pen_row[\'Accuracy (%)\']:.1f}%",\n    )\n    ax.set_xlabel("Average Output Tokens / Problem (Gen + Judge Tokens)", fontsize=10)\n    ax.set_ylabel("Exact-Match Accuracy (%)", fontsize=10)\n    ax.set_title("Accuracy vs. Total Output Tokens / Problem", fontsize=11, fontweight="bold")\n    ax.grid(True, linestyle="--", alpha=0.4)\n    ax.legend(loc="lower right", fontsize=8.5, frameon=True)\n    fig.tight_layout()\n    fig.savefig(os.path.join(fig_dir, "scaling_tokens.pdf"))\n    fig.savefig(os.path.join(fig_dir, "scaling_tokens.png"), dpi=200)\n    plt.close(fig)\n\n\ndef run_detailed_analysis(output_dir: str = "outputs", fig_dir: str = "figures") -> dict[str, Any]:\n    per_problem_path = os.path.join(output_dir, "per_problem.jsonl")\n    records = load_per_problem(per_problem_path)\n    with open(os.path.join(output_dir, "results_summary.json"), "r", encoding="utf-8") as f:\n        results_summary = json.load(f)\n    with open(os.path.join(output_dir, "aggregation_ablation.json"), "r", encoding="utf-8") as f:\n        ablation = json.load(f)\n    with open(os.path.join(output_dir, "sc_scaling_curve.json"), "r", encoding="utf-8") as f:\n        scaling_curve = json.load(f)\n\n    # 1. Bootstrap CIs across methods\n    method_keys = [\n        ("Greedy (N=1)", "greedy"),\n        ("Self-Consistency (N=5)", "sc_5"),\n        ("Compute-Matched SC (N=15)", "sc_15"),\n        ("PRM Best-of-5 (min)", "prm_min"),\n        ("PRM Best-of-5 (product)", "prm_product"),\n        ("PRM Best-of-5 (step_penalty, lambda=2.0)", "prm_step_penalty_2.0"),\n        ("PRM Best-of-5 (product_logprob)", "prm_product_logprob"),\n    ]\n    bootstrap_cis = {}\n    for label, pkey in method_keys:\n        flags = [\n            1 if r["predictions"].get(pkey) == r["ground_truth_answer"] else 0\n            for r in records\n        ]\n        mean_acc, ci_low, ci_high = compute_bootstrap_ci(flags, n_resamples=1000, seed=5326)\n        bootstrap_cis[label] = {\n            "mean": mean_acc,\n            "ci_95_low": ci_low,\n            "ci_95_high": ci_high,\n        }\n\n    # 2. Paired win/loss counts\n    paired_prm_vs_sc5 = compute_paired_win_loss(records, "prm_min", "sc_5")\n    paired_prm_vs_sc15 = compute_paired_win_loss(records, "prm_min", "sc_15")\n\n    # 3. Delimiter compliance rates\n    n_probs = len(records)\n    greedy_compliant = sum(1 for r in records if r["greedy"]["answer"] is not None)\n    all_sc15_cands = [c for r in records for c in r["candidates"]]\n    sc_cand_compliant = sum(1 for c in all_sc15_cands if c.get("answer") is not None)\n\n    # 4. 10x Greedy Compute Token Budget\n    avg_greedy_tokens = sum(r["bookkeeping"]["greedy_tokens"] for r in records) / n_probs\n    budget_10x_greedy = round(10.0 * avg_greedy_tokens, 2)\n\n    # 5. Mine qualitative failure cases and step pool for the 30-step Judge Audit\n    judge_fp_candidates = []\n    judge_fn_candidates = []\n    mode_collapse_cases = []\n    invalid_judged_steps = []\n    valid_judged_steps = []\n\n    for r in records:\n        prob_id = r["id"]\n        q = r["question"]\n        gt = r["ground_truth_answer"]\n        shared_5 = r.get("shared_candidates_5", r["candidates"][:5])\n\n        all_5_wrong = all(c.get("answer") != gt for c in shared_5)\n        if all_5_wrong:\n            mode_collapse_cases.append({\n                "id": prob_id,\n                "question": q,\n                "ground_truth_answer": gt,\n                "candidate_answers": [c.get("answer") for c in shared_5],\n                "candidate_0_steps": shared_5[0].get("steps", []) if shared_5 else [],\n                "candidate_0_scores": shared_5[0].get("step_scores", []) if shared_5 else [],\n            })\n\n        for c_idx, cand in enumerate(shared_5):\n            c_ans = cand.get("answer")\n            c_steps = cand.get("steps", [])\n            c_scores = cand.get("step_scores", [])\n            c_rats = cand.get("rationales", [])\n\n            # Candidate reached wrong answer, yet judge marked every single step VALID (+1.0)\n            if c_ans is not None and c_ans != gt and c_scores and all(s == 1.0 for s in c_scores):\n                judge_fp_candidates.append({\n                    "id": prob_id,\n                    "candidate_index": c_idx,\n                    "question": q,\n                    "ground_truth_answer": gt,\n                    "candidate_answer": c_ans,\n                    "steps": c_steps,\n                    "step_scores": c_scores,\n                    "rationales": c_rats,\n                })\n\n            # Candidate reached the right answer, yet judge marked at least one step INVALID (-1.0)\n            if c_ans == gt and c_scores and any(s == -1.0 for s in c_scores):\n                judge_fn_candidates.append({\n                    "id": prob_id,\n                    "candidate_index": c_idx,\n                    "question": q,\n                    "ground_truth_answer": gt,\n                    "candidate_answer": c_ans,\n                    "steps": c_steps,\n                    "step_scores": c_scores,\n                    "rationales": c_rats,\n                })\n\n            for s_idx, (st, sc) in enumerate(zip(c_steps, c_scores)):\n                rat = c_rats[s_idx] if s_idx < len(c_rats) else ""\n                step_entry = {\n                    "id": prob_id,\n                    "candidate_index": c_idx,\n                    "step_number": s_idx + 1,\n                    "question": q,\n                    "ground_truth_answer": gt,\n                    "candidate_answer": c_ans,\n                    "candidate_correct": (c_ans == gt),\n                    "preceding_steps": c_steps[:s_idx],\n                    "step_text": st,\n                    "judge_score": sc,\n                    "judge_verdict": "VALID" if sc > 0 else "INVALID",\n                    "judge_rationale": rat,\n                }\n                if sc < 0:\n                    invalid_judged_steps.append(step_entry)\n                else:\n                    valid_judged_steps.append(step_entry)\n\n    audit_dump = {\n        "total_judged_steps": len(invalid_judged_steps) + len(valid_judged_steps),\n        "total_invalid_judged_steps": len(invalid_judged_steps),\n        "total_valid_judged_steps": len(valid_judged_steps),\n        "judge_fp_candidates_count": len(judge_fp_candidates),\n        "judge_fn_candidates_count": len(judge_fn_candidates),\n        "mode_collapse_problems_count": len(mode_collapse_cases),\n        "judge_fp_examples": judge_fp_candidates[:15],\n        "judge_fn_examples": judge_fn_candidates[:15],\n        "mode_collapse_examples": mode_collapse_cases[:15],\n        "invalid_steps_sample": invalid_judged_steps[:30],\n        "valid_steps_sample": valid_judged_steps[:30],\n    }\n    with open(os.path.join(output_dir, "judge_audit_candidates.json"), "w", encoding="utf-8") as f:\n        json.dump(audit_dump, f, indent=2)\n\n    generate_plots(results_summary, scaling_curve, ablation, fig_dir)\n\n    report_stats = {\n        "num_problems": n_probs,\n        "bootstrap_cis": bootstrap_cis,\n        "paired_prm_min_vs_sc5": paired_prm_vs_sc5,\n        "paired_prm_min_vs_sc15": paired_prm_vs_sc15,\n        "greedy_delimiter_compliance_pct": round(100.0 * greedy_compliant / n_probs, 2),\n        "sampled_candidate_compliance_pct": round(\n            100.0 * sc_cand_compliant / max(1, len(all_sc15_cands)), 2\n        ),\n        "avg_greedy_tokens_per_problem": round(avg_greedy_tokens, 2),\n        "budget_10x_greedy_tokens_per_problem": budget_10x_greedy,\n    }\n    with open(os.path.join(output_dir, "report_analysis_stats.json"), "w", encoding="utf-8") as f:\n        json.dump(report_stats, f, indent=2)\n    return report_stats\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output-dir", type=str, default="outputs")\n    parser.add_argument("--fig-dir", type=str, default="figures")\n    args = parser.parse_args()\n    stats = run_detailed_analysis(args.output_dir, args.fig_dir)\n    print(json.dumps(stats, indent=2))\n',
}

for rel_path, content in FILES_TO_WRITE.items():
    with open(rel_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Verified/wrote {rel_path} ({len(content)} bytes)")


### Step 3: Verify Public Adapter Smoke Tests & Token Counting Tests

In [ ]:
!python3 tests/test_public_adapter_smoke.py
!python3 tests/test_token_counting.py

### Step 4: Inspect GPU Hardware & Run Full 50-Problem Benchmark Evaluation
- Uses `transformers.set_seed(5326 + problem_index)` before each problem.
- Incrementally checkpoints every completed problem to `outputs/per_problem.jsonl`.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

from run_all import (
    load_benchmark,
    load_existing_checkpoints,
    evaluate_single_problem,
    write_all_checkpoints,
    compute_and_save_artifacts,
)
from src.utils import load_models, make_hf_generate_fn, make_hf_judge_fn
import time

problems = load_benchmark('data/gsm8k_test_50.jsonl')
per_problem_path = 'outputs/per_problem.jsonl'
completed_map = load_existing_checkpoints(per_problem_path)
print(f'Loaded {len(problems)} problems. Existing checkpoints: {len(completed_map)}')

if len(completed_map) < len(problems):
    gen_model, gen_tokenizer, judge_model, judge_tokenizer = load_models(
        generator_device='cuda:0',
        judge_device='cuda:1',
    )
    gen_fn = make_hf_generate_fn(gen_model, gen_tokenizer)
    judge_fn = make_hf_judge_fn(judge_model, judge_tokenizer)
else:
    gen_fn = None
    judge_fn = None

ordered_records = []
for idx, prob in enumerate(problems):
    prob_id = prob['id']
    if prob_id in completed_map:
        rec = completed_map[prob_id]
        ordered_records.append(rec)
        print(f'[{idx+1:02d}/{len(problems):02d}] {prob_id} (restored from checkpoint)')
        continue
    t0 = time.perf_counter()
    rec = evaluate_single_problem(idx, prob, gen_fn, judge_fn)
    ordered_records.append(rec)
    completed_map[prob_id] = rec
    write_all_checkpoints(ordered_records, per_problem_path)
    dt = time.perf_counter() - t0
    preds = rec['predictions']
    print(f"[{idx+1:02d}/{len(problems):02d}] {prob_id} | GT={rec['ground_truth_answer']} | "
          f"Greedy={preds['greedy']} | SC@5={preds['sc_5']} | PRM(min)={preds['prm_min']} | "
          f"PRM(logprob)={preds['prm_product_logprob']} | SC@15={preds['sc_15']} ({dt:.1f}s)")

write_all_checkpoints(ordered_records, per_problem_path)
summary, ablation, curve = compute_and_save_artifacts(ordered_records, 'outputs')


### Step 5: Statistical Analysis, Scaling Plots & Downloadable Archive

In [ ]:
import json
import shutil
from analyze_outputs import run_detailed_analysis

stats = run_detailed_analysis('outputs', 'figures')
print('=== Summary Table ===')
with open('outputs/results_summary.json') as f:
    print(json.dumps(json.load(f), indent=2))

print('\n=== Aggregation Ablation ===')
with open('outputs/aggregation_ablation.json') as f:
    print(json.dumps(json.load(f), indent=2))

print('\n=== Report Stats (Bootstrap CIs & Win/Loss) ===')
print(json.dumps(stats, indent=2))

# Bundle outputs/ and figures/ into a single zip file in /kaggle/working for 1-click download
zip_target = '/kaggle/working/pa3_kaggle_outputs' if os.path.exists('/kaggle/working') else 'pa3_kaggle_outputs'
shutil.make_archive(zip_target, 'zip', '.', 'outputs')
print(f'\nCreated downloadable archive: {zip_target}.zip')